# Importation des bibliothèques

In [ ]:
import pandas as pd
from utils import  beau_tableau # Fonction pour la mise en forme des tableaux. Voir ./utils.py



# Importation du dataset

In [ ]:
# Importation
df = pd.read_csv(
 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb' 
)
# Aperçu
print(f"Dimension du dataset : {df.shape}")
print(f"Type des variables  : {df.dtypes}")

print("Aperçu du dataset  : ")
print(df.head(2))


On va spécifier les types des variables pour un travail plus fluide.

In [ ]:
df = df.convert_dtypes()
df.dtypes
df["code_commune"] = df["code_commune"].astype("string")


# 0. Explorations de la base de données

In [ ]:
# Aperçu de la base de données
beau_tableau(df.head(),
col_gras="all"
)

**Avant d'attaquer la question 1, on décide d'en savoir plus sur les communes et les départements pour mieux construire le code_commune.**

In [ ]:
print(df['code_commune'].str.len().value_counts())
print(df['code_departement'].str.len().value_counts())
# Voir  les codes départements uniques
print(df['code_departement'].unique())
#beau_tableau(df[df['code_departement'].str.len() == 11])
print(df[df['code_departement'].str.len() == 11]['code_departement'].unique())


On remarque que les codes des communes sont constitués de chiffres avec 3 chiffes au maximum. Pour les départements on a au maximum une chaine de 11 caractères correspondant à fr_etranger. 

In [ ]:
# Départements avec 3 chiffres
print(df[df['code_departement'].str.len() == 3])
#df[df['code_departement'] == "988"]

# Type de commune avec départements de 3 chiffres 
print(df[df['code_departement'].str.len() == 3]['code_commune'].str.len().value_counts())


In [ ]:
# # Départements de 3 chiffres dont les codes de commune n'ont pas trois chiffres
mask = (df['code_departement'].str.len() == 3) & (df['code_commune'].str.len() != 3)
df[mask][['code_departement', 'code_commune', 'libelle_commune']].drop_duplicates().head(20)

In [ ]:
# Départements de 3 chiffres dont les codes de commune ne commencent pas par le dernier chiffre du code département
# Le but est de savoir si on peut extraire les 2 premiers chiffres du code département dans tous les cas
mask = (df['code_departement'].str.len() == 3) & (df['code_commune'].str.len() == 3)
df_dom = df[mask].copy()

df_dom['dept_last'] = df_dom['code_departement'].str[-1:]
df_dom['commune_first'] = df_dom['code_commune'].str[0]

df_dom[df_dom['dept_last'] != df_dom['commune_first']][['code_departement', 'code_commune', 'dept_last', 'commune_first']].drop_duplicates().head(20)



Après une vérification initiale des formats, on observe que `code_commune` comporte jusqu'à 3 chiffres et que `code_departement` s'étend de 2 à `fr_etranger` (11), ce qui rend une normalisation nécessaire. Il faut ensuite extraire les cas particuliers, notamment les départements à 3 chiffres (Dom-Tom, etc.) et les communes qui ne suivent pas systématiquement la logique « 2 premiers chiffres du département + code commune ». La recherche de contre-exemples (pour les cas à 3 chiffres de département et 3 chiffres de commune où le premier chiffre diffère du dernier chiffre du département) n'en révèle aucun, ce qui souligne toutefois la complexité d'établir une règle unique. En conclusion, il convient de traiter explicitement les cas particuliers (ex. `fr_etranger` -> `99`, Wallis-et-Futuna, etc.), puis d'harmoniser `code_commune` via `zfill` et une découpe contrôlée afin de garantir des codes cohérents.


# 1. Explorations générales

## Question 1 :  Création et mise à jour des variables

Création ou mise à jours des  variables :
- `code_commune` : département + commune
- `candidat` : prénom + nom 

### Code commune

In [ ]:
#  Remplacer fr_etranger par 99
df['code_departement'] = df['code_departement'].replace('fr_etranger', '99')

# Construire le code commune
df['code_commune'] = (
    df['code_departement'].str[:2]
    + df['code_commune'].str[-3:].str.zfill(3) # pour completer des zeros avant le chiffre pour en avoir 3
)

# Correction Wallis-et-Futuna (986) : on prend les 3 chiffres du département
mask_wallis = df['code_departement'].astype(str) == '986'
df.loc[mask_wallis, 'code_commune'] = (
    '986' + df.loc[mask_wallis, 'code_commune'].str[-1:].str.zfill(2)
)

# Vérification
df['code_commune'].str.len().value_counts()

### Création de candidat

In [ ]:
df['candidat'] = df['prenom'] + ' ' + df['nom']
df.sample(7)

## Question 2 : Complétons la phrase grâce à Python

In [ ]:
# Préliminaires
print(df[df['prenom'].isna()]['nom'].unique()) # toutes les catégories de NA

candidats = df[df['prenom'].notna()]['candidat'].nunique()#on y enlève les na



*On peut alors écrire la phrase*

In [ ]:
# Phrase
f"En 2022, il y avait {candidats} candidats à l'élection présidentielle."

## Question 3

In [ ]:
# Votes exprimés uniquement (on exclut les lignes sans prénom)
df_exprimes = df[df['prenom'].notna()]

# Total des voix par candidat
votes_national = df_exprimes.groupby('candidat')['voix'].sum().reset_index()
votes_national.columns = ['Candidat', 'Nombre votes (total)']

# Total général pour calculer les pourcentages
total_exprimes = votes_national['Nombre votes (total)'].sum()

# Score en %
votes_national['Score (% votes exprimés)'] = (votes_national['Nombre votes (total)'] / total_exprimes * 100).round(2)

# Tri décroissant
votes_national = votes_national.sort_values('Nombre votes (total)', ascending=False).reset_index(drop=True)

votes_national.head()

*On obtient quelque chose de potable. On passee à la mise en forme.*

### Tableau mis en forme

In [ ]:

beau_tableau(
    votes_national,
    col_entiers=["Nombre votes (total)"],
    col_pourcentages=["Score (% votes exprimés)"],
    titre="Elections",
    sous_titre="Résultats du premier tour",
    date="10 avril 2022",
    note="Table 1. Résultats du premier tour de l'élection présidentielle 2022"
)

**Interprétation du tableau des résultats du premier tour de l'élection présidentielle 2022**
Le tableau montre les candidats classés par nombre de voix décroissant, avec le total des voix et le pourcentage des votes exprimés. Emmanuel Macron arrive en tête avec environ 9.78 millions de voix (27.85%), suivi de Marine Le Pen avec environ 8.13 millions (23.15%). Les autres candidats ont des scores inférieurs, avec Jean-Luc Mélenchon à environ 7.71 millions (21.95%).
Ce tableau représente les résultats nationaux agrégés, excluant les votes blancs ou nuls.